<a href="https://colab.research.google.com/github/Pallavi20004/Demo/blob/main/TASK9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from numpy import zeros, ones
from numpy.random import randn, randint
from keras.datasets.cifar10 import load_data
from keras.optimizers import Adam
from keras.models import Sequential
from keras.layers import Dense, Reshape, Flatten, Conv2D, Conv2DTranspose
from keras.layers import LeakyReLU, Dropout
from matplotlib import pyplot as plt
import numpy as np
import tensorflow as tf

# Check if GPU is available
def check_gpu():
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print("✅ GPU is available!")
    else:
        print("⚠️ Using CPU, training might be slow.")

check_gpu()

# Load dataset
(trainX, _), (_, _) = load_data()
trainX = (trainX.astype('float32') - 127.5) / 127.5  # Normalize images

def define_discriminator(in_shape=(32,32,3)):
    model = Sequential()
    model.add(Conv2D(128, (3,3), strides=(2,2), padding='same', input_shape=in_shape))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Conv2D(128, (3,3), strides=(2,2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Flatten())
    model.add(Dropout(0.4))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5), metrics=['accuracy'])
    return model

def define_generator(latent_dim):
    model = Sequential()
    model.add(Dense(128 * 8 * 8, input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Reshape((8, 8, 128)))
    model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Conv2D(3, (3,3), activation='tanh', padding='same'))
    return model

def define_gan(generator, discriminator):
    discriminator.trainable = False
    model = Sequential([generator, discriminator])
    model.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))
    return model

def generate_real_samples(dataset, n_samples):
    ix = randint(0, dataset.shape[0], n_samples)
    X = dataset[ix]
    y = ones((n_samples, 1))
    return X, y

def generate_latent_points(latent_dim, n_samples):
    x_input = randn(latent_dim * n_samples).reshape(n_samples, latent_dim)
    return x_input

def generate_fake_samples(generator, latent_dim, n_samples):
    X = generator.predict(generate_latent_points(latent_dim, n_samples))
    y = zeros((n_samples, 1))
    return X, y

def train(g_model, d_model, gan_model, dataset, latent_dim, n_epochs=50, n_batch=64):
    bat_per_epo = int(dataset.shape[0] / n_batch)
    half_batch = n_batch // 2

    for i in range(n_epochs):
        for j in range(bat_per_epo):
            X_real, y_real = generate_real_samples(dataset, half_batch)
            d_loss_real, _ = d_model.train_on_batch(X_real, y_real)

            X_fake, y_fake = generate_fake_samples(g_model, latent_dim, half_batch)
            d_loss_fake, _ = d_model.train_on_batch(X_fake, y_fake)

            X_gan = generate_latent_points(latent_dim, n_batch)
            y_gan = ones((n_batch, 1))
            g_loss = gan_model.train_on_batch(X_gan, y_gan)

            print(f'Epoch {i+1}, Batch {j+1}/{bat_per_epo}, d_real={d_loss_real:.3f}, d_fake={d_loss_fake:.3f}, g={g_loss:.3f}')

        if (i+1) % 10 == 0:
            g_model.save(f'generator_epoch_{i+1}.h5')
            d_model.save(f'discriminator_epoch_{i+1}.h5')
            print(f'✔️ Models saved at epoch {i+1}')

latent_dim = 100
discriminator = define_discriminator()
generator = define_generator(latent_dim)
gan_model = define_gan(generator, discriminator)
train(generator, discriminator, gan_model, trainX, latent_dim, n_epochs=50)

from keras.models import load_model

def show_plot(examples, n):
    for i in range(n * n):
        plt.subplot(n, n, 1 + i)
        plt.axis('off')
        plt.imshow((examples[i] + 1) / 2.0)  # Rescale images
    plt.show()

model = load_model('generator_epoch_50.h5')
latent_points = generate_latent_points(100, 25)
X = model.predict(latent_points)
show_plot(X, 5)


⚠️ Using CPU, training might be slow.
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/trainer.py:82: UserWarning: The model does not have any trainable weights.
 

Streaming output truncated to the last 5000 lines.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step
Epoch 1, Batch 247/781, d_real=1.199, d_fake=1.202, g=0.247
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step
Epoch 1, Batch 248/781, d_real=1.201, d_fake=1.203, g=0.247
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step
Epoch 1, Batch 249/781, d_real=1.202, d_fake=1.205, g=0.246
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step
Epoch 1, Batch 250/781, d_real=1.204, d_fake=1.206, g=0.246
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step
Epoch 1, Batch 251/781, d_real=1.205, d_fake=1.208, g=0.245
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 332ms/step
Epoch 1, Batch 252/781, d_real=1.207, d_fake=1.209, g=0.244
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 334ms/step
Epoch 1, Batch 253/781, d_real=1.208, d_fake=1.211, g=0.244
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step
Epoch 1, Batch 254/781, d_real=1.210, d_fake=1.212, g=0.243
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step
Epoch 1, Batch 255/781, d_real=1.211, d_fake=1.213, g=0.242
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step
Epoch 1, Batch 256/

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

